In [7]:
from jugaad_data.nse import NSELive
n = NSELive()
status = n.market_status()
status['marketState']

[{'market': 'Capital Market',
  'marketStatus': 'Closed',
  'tradeDate': '28-Apr-2025 15:30',
  'index': 'NIFTY 50',
  'last': 24328.5,
  'variation': 289.15000000000146,
  'percentChange': 1.2,
  'marketStatusMessage': 'Normal Market has Closed'},
 {'market': 'Currency',
  'marketStatus': 'Closed',
  'tradeDate': '28-Apr-2025',
  'index': '',
  'last': '',
  'variation': '',
  'percentChange': '',
  'marketStatusMessage': 'Market is Closed'},
 {'market': 'Commodity',
  'marketStatus': 'Open',
  'tradeDate': '28-Apr-2025',
  'index': '',
  'last': '',
  'variation': '',
  'percentChange': '',
  'marketStatusMessage': 'Market is Open'},
 {'market': 'Debt',
  'marketStatus': 'Closed',
  'tradeDate': '28-Apr-2025',
  'index': '',
  'last': '',
  'variation': '',
  'percentChange': '',
  'marketStatusMessage': 'Market is Closed'},
 {'market': 'currencyfuture',
  'marketStatus': 'Closed',
  'tradeDate': '28-Apr-2025',
  'index': '',
  'last': '85.1850',
  'variation': '',
  'percentChange':

In [2]:
tick_data = n.tick_data("HDFC")

In [3]:
tick_data['grapthData'][0:10]


[]

In [8]:
option_chain = n.index_option_chain("NIFTY") # Index Option chains
eq_option_chain = n.equities_option_chain("RELIANCE") # Equity option chains
curr_option_chain = n.currency_option_chain("USDINR") # Currency option chains

In [44]:
option_chain['records']['underlyingValue']

24328.5

In [14]:
testDF = option_chain['filtered']['data']

In [ ]:
testDF

In [39]:
count =0 
for i in testDF:
    if(i['CE']['openInterest']>50000 or i['PE']['openInterest']>50000): 
        print(i['strikePrice'])
        count=count+1
print(count)

20400
21000
21500
22000
22500
23000
23200
23500
23800
23900
24000
24100
24200
24300
24350
24400
24500
24600
24700
24800
24900
25000
25100
25200
25500
25800
26000
26100
28


In [21]:

for option in testDF :
    print( "{}\t{}\t::\t{}\t::\t{}\t{}".format(option['CE']['openInterest'],option['CE']['lastPrice'], option['strikePrice'], option['PE']['lastPrice'],option['PE']['openInterest']))
    # print(option['CE']['openInterest'])



0	0	::	20400	::	1.75	73755
0	0	::	20450	::	1.7	20909
11	3850	::	20500	::	1.9	35912
1	3343.15	::	20550	::	1.8	1696
1	3281.55	::	20600	::	2.25	15358
0	0	::	20650	::	1.8	907
1	3180.25	::	20700	::	2.25	15183
1	3423.25	::	20750	::	2.1	1030
1	3073.6	::	20800	::	2.25	14747
0	0	::	20850	::	2.65	539
2	3345	::	20900	::	2.55	16213
1	2924.85	::	20950	::	2.15	984
68	3006.65	::	21000	::	2.3	71641
1	2825.4	::	21050	::	2	3297
1	3190.35	::	21100	::	2.7	6422
2	3026.1	::	21150	::	2.9	2601
0	2940	::	21200	::	2.85	6695
2	2926.8	::	21250	::	3.25	2105
4	2873.05	::	21300	::	2.95	4570
4	2823.45	::	21350	::	3.05	1611
9	2940	::	21400	::	2.75	3832
9	2718.2	::	21450	::	3.3	1554
281	2843.35	::	21500	::	3.1	62416
8	2633.35	::	21550	::	3	1950
197	2727.95	::	21600	::	3.4	10290
7	2535.4	::	21650	::	3.6	2529
40	2632	::	21700	::	4.05	14612
11	2431.75	::	21750	::	4.45	4302
127	2550	::	21800	::	4.1	16537
10	2528.9	::	21850	::	4.05	3914
18	2451.15	::	21900	::	4.3	14633
6	1927.65	::	21950	::	4.5	3462
2720	2348.95	::	22000	::

In [1]:
from helpers.nse_data import NSEData
from helpers.FnO import analyse_option_chain, get_next_expiry_date

In [28]:
def myplotsOptionChain(r):
    df_data = {'contract_type':[],'expiryDate': [],'strikePrice':[],'openInterest': [],'changeinOpenInterest': [],'pchangeinOpenInterest': [],'totalTradedVolume': [], 
               'totalBuyQuantity': [], 'totalSellQuantity': [], 'change':[]}
    keys = df_data.keys()
    data = r['records']['data']

    for entry in data:
        if entry.get('CE'):
            [df_data[name].append(entry['CE'][name]) if name!='contract_type' else df_data['contract_type'].append('Calls_CE') for name in keys]

        if entry.get('PE'):
            [df_data[name].append(entry['PE'][name]) if name!='contract_type' else df_data['contract_type'].append('Puts_PE') for name in keys]
        

    df = pd.DataFrame(df_data)
    df.rename(columns = {'expiryDate':'expiry_date', 'strikePrice':'strike_price'},inplace = True)
    df['expiry_date'] = df['expiry_date'].apply(lambda x:datetime.strptime(x, "%d-%b-%Y").strftime("%d-%b-%Y"))
    df['strike_price'] = df['strike_price'].apply(lambda x: int(x))
    df['absChangeOI'] = df['changeinOpenInterest'].apply(lambda x: abs(x))
    df['absChange'] = df['change'].apply(lambda x: abs(x))
    
    expiry_dates= ['30-Apr-2025']
    # Get specific expiry date
    if not expiry_dates:
        recent_expiry = get_next_expiry_date()[0]
        sup_plot_text_date = recent_expiry
        expiry_dates = [recent_expiry] # Single element tuple requires ,
    else:
        recent_expiry = expiry_dates[0]
        sup_plot_text_date = 'All Available Expiries'
        expiry_dates = [datetime.strptime(x, "%d-%b-%Y").strftime("%d-%b-%Y") for x in expiry_dates]   
    df = df[df['expiry_date'].isin(expiry_dates)]
    
    Plots.plot_Option_chain("symbol", df, compare_with, top_n, sup_plot_text_date, fig_size=fig_size)

    return df

In [29]:
import pandas as pd
from datetime import datetime
from helpers.plotting import Plots 
import matplotlib.pyplot as plt
import seaborn as sns
myplotsOptionChain(option_chain)

NameError: name 'compare_with' is not defined

In [ ]:
  analyse_option_chain('TATACHEM', plot = True, fig_size=(21,6.7)) # Read the Doc String. Manually compare other parameters such as openInterest

https://www.nseindia.com/api/option-chain-equities?symbol=TATACHEM


JSONDecodeError: Expecting value: line 1 column 1 (char 0)